In [1]:
import os 
import sys
sys.path.append('../src')

from preprocessing import *

In [2]:
dfs = get_dfs(os.path.dirname(os.getcwd()))
static_df = create_static_df(dfs)

In [3]:
ca, lab = create_vitals_df(dfs)
med = create_medication_df(dfs)

Unique patients in clinical assessments: 3465
Removing patients that are not in static_df
Unique patients in clinical assessments: 3423
Average entries per patient 61.16155419222904
Unique patients in lab df: 3460
Removing patients that are not in static_df
Unique patients in lab df: 3410
Average entries per patient 472.2140762463343
Unique patients in medication: 3335
Removing patients that are not in static_df
Unique patients in clinical assessments: 3296
Average entries per patient 78.9


In [10]:
lab[lab['description'] == 'AlbuminKSU']['unit'].value_counts()
x =lab[lab['description'] == 'AlbuminKSU']

In [8]:
ts_data = create_ts_data(ca, vitals_lab=lab, medication=med, merge_lab=True, merge_med=False)
ts_data

,patient_id,transplant_id,rel_days,bp_sys,bp_dia,weight,urine_volume,hr,temperature,diuresis_time,creatinine,leukocyte
0,33,1805,81,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,33,1805,84,168.0,121.0,55,NaN,86.0,NaN,NaN,1.06,NaN
2,33,1805,89,132.0,95.0,55,NaN,87.0,NaN,NaN,1.09,NaN
3,33,1805,101,136.0,100.0,58,NaN,91.0,"36,1",NaN,1.14,500.0
4,33,1805,109,141.0,103.0,60,NaN,67.0,NaN,NaN,1.00,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
381105,38658,15328,1017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.38,NaN
381106,38658,15328,1101,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.48,NaN
381107,38658,15328,1192,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.38,NaN
381108,38658,15328,1288,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.29,NaN


In [4]:
ca.columns

Index(['patient_id', 'transplant_id', 'rel_days', 'bp_sys', 'bp_dia', 'weight',
       'urine_volume', 'hr', 'temperature', 'diuresis_time'],
      dtype='object')

In [6]:
print(len(dfs['lab_cohort']))
dfs['lab_cohort']['Bezeichnung'].value_counts().head(15)

1657618


Bezeichnung
KreatininHP    336909
LeukoEB        332109
CRPHP          249930
LeukoTUR       155696
ProteinTUR     155605
ProteinCSU     113059
ProteinDSU     108946
AlbuminKSU      53010
ProteinKSU      42113
ProteinKUR      41178
AlbuminKUR      34631
PROTkU          24408
CRPRocheHP       2653
GEIW             1777
CRP              1692
Name: count, dtype: int64

In [5]:
# create vitals from clinical assessments 
print(dfs['clinical_assessment'].columns)
print(dfs['clinical_assessment'].count())
# no data points for Infektion, Zielwerte, Uhrzeit

Index(['VerlaufID', 'PatientID', 'TransplantationID', 'DateofTransplantation',
       'Datum', 'Date of graft loss', 'OPDtime', 'HatBeurteilung',
       'Beurteilung', 'HatBeurteilungAerztlich', 'BeurteilungAerztlich',
       'HatBeurteilungIntern', 'BeurteilungIntern', 'Blutdruck_systolisch',
       'Blutdruck_diastolisch', 'Gewicht', 'Urinvolumen', 'naechster_Termin',
       'Herzfrequenz', 'Infektion', 'Temperatur', 'Zielwert_Bezeichnung',
       'Zielwert_Wert', 'Zielwert_Einheit', 'Diuresezeit',
       'naechster_TerminZeit', 'Quelle', 'Gesehen_durch', 'Therapie_durch',
       'Uhrzeit', 'naechster_TerminArt', 'OPDGFtime'],
      dtype='object')
VerlaufID                  215137
PatientID                  215137
TransplantationID          215137
DateofTransplantation      215137
Datum                      215137
Date of graft loss          38964
OPDtime                    215137
HatBeurteilung             215137
Beurteilung                 89859
HatBeurteilungAerztlich    215137
B

In [6]:
vitals_ca = dfs['clinical_assessment'][['PatientID', 'TransplantationID', 'OPDtime', 'Blutdruck_systolisch', 'Blutdruck_diastolisch', 'Gewicht', 'Urinvolumen', 'Herzfrequenz', 'Temperatur', 'Diuresezeit']].rename(
        columns={
            'PatientID': 'patient_id',
            'TransplantationID': 'transplant_id',
            'OPDtime': 'rel_days', # date of assessment in days after transplantation
            'Blutdruck_systolisch': 'bp_sys',
            'Blutdruck_diastolisch': 'bp_dia',
            'Gewicht': 'weight',
            'Urinvolumen': 'urine_volume',  # lower urine volume indicates issues with kidney function
            'Herzfrequenz': 'hr',
            'Temperatur': 'temperature', 
            'Diuresezeit': 'diuresis_time',  # duration over which urine output is measured
        }
    )

print(f"Unique patients in clinical assessments: {vitals_ca['patient_id'].nunique()}")
print('Removing patients that are not in static_df')
vitals_ca = vitals_ca.merge(
    static_df[['patient_id', 'transplant_id']],
    how='inner',  # Inner join to keep only matching entries
    on=['patient_id', 'transplant_id']
)

print(f"Unique patients in clinical assessments: {vitals_ca['patient_id'].nunique()}")
print(f"Average entries per patient {len(vitals_ca) / vitals_ca['patient_id'].nunique()}")

Unique patients in clinical assessments: 3465
Removing patients that are not in static_df
Unique patients in clinical assessments: 3423
Average entries per patient 61.16155419222904


In [7]:
vitals_ca

,patient_id,transplant_id,rel_days,bp_sys,bp_dia,weight,urine_volume,hr,temperature,diuresis_time
0,33,1805,"81,00",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,33,1805,"84,00",168.0,121.0,55,NaN,86.0,NaN,NaN
2,33,1805,"89,00",132.0,95.0,55,NaN,87.0,NaN,NaN
3,33,1805,"101,00",136.0,100.0,58,NaN,91.0,"36,1",NaN
4,33,1805,"109,00",141.0,103.0,60,NaN,67.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
209351,38071,15166,"1478,00",149.0,89.0,47,NaN,66.0,NaN,NaN
209352,38071,15166,"1485,00",NaN,NaN,NaN,NaN,NaN,NaN,NaN
209353,38081,15319,"2866,00",NaN,NaN,NaN,NaN,NaN,NaN,NaN
209354,38081,15319,"3151,00",NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
print(dfs['lab_cohort'].columns)
print(dfs['lab_cohort'].count())

Index(['LaborID', 'PatientID', 'TransplantationID', 'LabDate', 'Labtime',
       'Abnahmezeit', 'Bezeichnung', 'Wert', 'Einheit', 'BeNorm', 'MatNorm',
       'Bez', 'LOINC'],
      dtype='object')
LaborID              1657618
PatientID            1657618
TransplantationID    1657618
LabDate              1657618
Labtime              1657618
Abnahmezeit          1636502
Bezeichnung          1657618
Wert                 1654538
Einheit              1345795
BeNorm               1657618
MatNorm              1657618
Bez                  1657618
LOINC                  11955
dtype: int64


In [9]:
vitals_lab = dfs['lab_cohort'][['PatientID', 'TransplantationID', 'Labtime', 'Bezeichnung', 'Wert', 'Einheit']].rename(
        columns={
            'PatientID': 'patient_id',
            'TransplantationID': 'transplant_id',
            'Labtime': 'rel_days', # date of assessment in days after transplantation
            'Bezeichnung': 'description',
            'Wert' : 'value',
            'Einheit': 'unit',
        }
    )

print(f"Unique patients in lab df: {vitals_lab['patient_id'].nunique()}")
print('Removing patients that are not in static_df')
vitals_lab = vitals_lab.merge(
    static_df[['patient_id', 'transplant_id']],
    how='inner',  # Inner join to keep only matching entries
    on=['patient_id', 'transplant_id']
)

print(f"Unique patients in lab df: {vitals_lab['patient_id'].nunique()}")
print(f"Average entries per patient {len(vitals_lab) / vitals_lab['patient_id'].nunique()}")

Unique patients in lab df: 3460
Removing patients that are not in static_df
Unique patients in lab df: 3410
Average entries per patient 472.2140762463343
